# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
print('shape: ', df.shape)
print('dtypes:')
print(df.dtypes)
print('nulls: ')
print(df.isnull().sum())
print('dupes: ', df.duplicated().sum())

shape:  (8, 6)
dtypes:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
nulls: 
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
dupes:  1


**What is wrong with this data?** List at least five specific problems:

1. Category labels are inconsistent, like Food vs food or RainGear vs rain-gear, meaning they won't group together until normalized.
2. Theres an exact duplicate row, order 1 appears twice.
3. A negative quantity, which doesn't make sense for there to be -3 units sold. Maybe it could be a refund but there is no specification about a data entry error or something.
4. price is an object (text) not a number, where some values have $ included in them and some don't, so the mixed formatting reads it as object. This is a problem because you can't do numerical calculations on a string.
5. There are missing values in 3 different columns, qty for order 3, ts for order 6, and item for order 7.

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean =  df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied

# TODO:
log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# TODO:
clean['price'] = (
    clean['price']
    .astype(str)
    .str.strip()
    .str.replace('$', '')
    .str.replace(',', '')
    .astype(float)
)

assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text
log('price_to_float', 'stripped $ and whitespace, cast price to float', len(clean))

[price_to_float] stripped $ and whitespace, cast price to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
# TODO: clean['qty'] = pd.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately

# Decision 1: no quantity
# A quantity is required to calculate revenue at all. There's no value
# to fill in, so keeping it would just mean carrying a row that can
# never contribute a real number to any total.
clean = clean.dropna(subset=['qty']).copy()
log('drop_missing_qty', 'dropped rows with no quantity recorded', missing)

# Decision 2: negative quantity
# treat as a refund and keep it, rather than dropping it or flipping
# its sign. A negative qty likely is a return, which should reduce reported
# units and revenue, not disappear and not get treated as a positive
# sale (which would overstate revenue by double-counting units
# that came back).
log('drop_negative_qty', 'kept rows with negative quantity, no drop or flip', negative)


[drop_missing_qty] dropped rows with no quantity recorded (1 row(s))
[drop_negative_qty] kept rows with negative quantity, no drop or flip (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))

before_count = len(clean['category'].unique())

# TODO: lowercase, strip, remove punctuation
clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace('-', '', regex=False)
)
# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'food': 'Food',
    'apparel': 'Apparel',
    'merch': 'Merch',
    'raingear': 'RainGear',
}
clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

after_count = clean['category'].nunique()
log('normalize_category', f'lowercased/stripped punctuation, mapped variants to canonical names ({before_count} -> {after_count} categories)', len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'RainGear']
[normalize_category] lowercased/stripped punctuation, mapped variants to canonical names (6 -> 4 categories) (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
# TODO
print('before:', sorted(clean['item'].dropna().unique()))

before_count = clean['item'].nunique()

# normalize case and whitespace first
clean['item'] = clean['item'].str.lower().str.strip()

# map the remaining spelling variant explicitly
ITEM_MAP = {
    'cheese burger': 'cheeseburger',
    'cheeseburger': 'cheeseburger',
    'rain poncho': 'rain poncho',
}
clean['item'] = clean['item'].map(ITEM_MAP).fillna(clean['item'])

# title-case for a readable final label
clean['item'] = clean['item'].str.title()

print('after: ', sorted(clean['item'].dropna().unique()))

after_count = clean['item'].nunique()

log('normalize_item_spelling',
    f'lowercased/mapped spelling variants to canonical names ({before_count} -> {after_count} items)',
    len(clean))

# The row with no item at all (order 7): decide what to do
missing_item = clean['item'].isna().sum()

# Decision: drop. Unlike category or price, there's no reasonable value
# to infer for a completely unnamed product — a blank item can't be counted
# toward any product-level total (e.g. "how many Rain Ponchos sold"), and
# keeping it would just leave an unlabeled row cluttering every group-by.
clean = clean.dropna(subset=['item']).copy()
log('drop_missing_item', 'dropped row(s) with no item name recorded', missing_item)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'Rain Poncho', 'Uva T-Shirt']
[normalize_item_spelling] lowercased/mapped spelling variants to canonical names (5 -> 3 items) (6 row(s))
[drop_missing_item] dropped row(s) with no item name recorded (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
# TODO
before_nat = clean['ts'].isna().sum()

clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed = clean['ts'].isna().sum() - before_nat
print('timestamps that failed to parse:', failed)
print('total NaT after parsing (including originally-missing):', clean['ts'].isna().sum())

log('parse_timestamps', 'parsed ts to datetime, coercing unparseable values to NaT', failed)

clean['hour'] = clean['ts'].dt.hour

log('add_hour_column', 'derived hour column from parsed ts', len(clean))

timestamps that failed to parse: 2
total NaT after parsing (including originally-missing): 3
[parse_timestamps] parsed ts to datetime, coercing unparseable values to NaT (2 row(s))
[add_hour_column] derived hour column from parsed ts (5 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [10]:
# TODO: assertions
assert clean.duplicated().sum() == 0, 'duplicates should be gone'
assert clean['price'].dtype == float, 'price should be numeric'
assert pd.api.types.is_numeric_dtype(clean['qty']), 'qty should be numeric'
assert clean['category'].isin(['Food', 'Merch', 'Apparel', 'RainGear']).all(), 'category should only contain canonical values'
assert clean['item'].isna().sum() == 0, 'no row should be missing an item name'
assert clean['qty'].isna().sum() == 0, 'no row should be missing a quantity after cleaning'

# TODO: clean['revenue'] = ...
clean['revenue'] = clean['price'] * clean['qty']
# TODO: print rows, units, revenue, distinct categories
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 6.0
revenue: 76.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price_to_float,"stripped $ and whitespace, cast price to float",7
2,drop_missing_qty,dropped rows with no quantity recorded,1
3,drop_negative_qty,"kept rows with negative quantity, no drop or flip",1
4,normalize_category,"lowercased/stripped punctuation, mapped varian...",6
5,normalize_item_spelling,lowercased/mapped spelling variants to canonic...,6
6,drop_missing_item,dropped row(s) with no item name recorded,1
7,parse_timestamps,"parsed ts to datetime, coercing unparseable va...",2
8,add_hour_column,derived hour column from parsed ts,5


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
rows_after = len(clean)            # TODO
revenue_after = clean['revenue'].sum()        # TODO
biggest_decision = 'drop_negative_qty (kept the refund row instead of dropping or flipping its sign)'    # TODO: which choice moved the number most
revenue_other_way = revenue_after - clean.loc[clean['qty'] < 0, 'revenue'].sum()     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: drop_negative_qty (kept the refund row instead of dropping or flipping its sign)
revenue the other way: 94.5
